In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install -q -U langchain langchain-community langchain-core langchain-classic transformers accelerate pyngrok streamlit pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 105.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 94.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 102.0 MB/s eta 0:00:0000:010:01
   ━━

In [2]:
!pip install -q "transformers==4.41.0" accelerate bitsandbytes streamlit pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 71.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.1 MB/s eta 0:00:0000:01


In [3]:
import shutil, os
cache = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(cache):
    for folder in os.listdir(cache):
        if any(x in folder.lower() for x in ["phi", "mistral", "tinyllama"]):
            shutil.rmtree(os.path.join(cache, folder))
            print("Deleted:", folder)
print("Done")

Done


In [4]:
import torch
import json
import re
import subprocess
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_core.prompts import PromptTemplate

In [5]:
study_template = """Analyze the student's data and create a structured study plan. 
You must respond with ONLY a valid JSON block containing a list of study sessions. Do not include any intro or outro text.

Expected JSON format:
{{
  "plan": [
    {{
      "day": "Day and date of the session",
      "subject": "Subject name",
      "task_type": "Type of study (e.g., Revision, Solving Exams, Reading concepts)",
      "duration": "Suggested time in hours"
    }}
  ]
}}

Student Data:
- Subjects: {subjects}
- Exam Dates: {exam_dates}
- Current Level: {student_level}

JSON:"""

prompt_template = PromptTemplate(
    template=study_template,
    input_variables=["subjects", "exam_dates", "student_level"]
)

In [6]:
def generate_study_plan_chain(subjects, exam_dates, student_level):
    formatted_prompt = prompt_template.format(
        subjects=subjects,
        exam_dates=exam_dates,
        student_level=student_level
    )
    
    raw_response = generate_text(formatted_prompt, max_new_tokens=400)
    print("RAW OUTPUT:", repr(raw_response))
    
    # Extracting JSON files
    match = re.search(r'\{.*?\}', raw_response, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
            
    # Fallback في حال فشل البارسر (حتى لا يتوقف التطبيق)
    return {
        "plan": [
            {
                "day": "خطأ في التنسيق",
                "subject": "يرجى إعادة المحاولة",
                "task_type": raw_response[:100],
                "duration": "0"
            }
        ] 
    }

Stream Lit

In [9]:
app_code = (
    'import streamlit as st\n'
    'import pandas as pd\n'
    'import torch, json\n'
    'from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n'
    '\n'
    'st.set_page_config(page_title="AI Study Planner", layout="wide", page_icon="📚")\n'
    'st.title("📚 منظم المذاكرة الذكي - AI Study Planner")\n'
    'st.write("أدخل بياناتك الدراسية ودع الذكاء الاصطناعي يصمم لك جدولك المثالي.")\n'
    '\n'
    'MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"\n'
    '\n'
    'if "result_df" not in st.session_state:\n'
    '    st.session_state.result_df = None\n'
    'if "raw_output" not in st.session_state:\n'
    '    st.session_state.raw_output = None\n'
    '\n'
    '@st.cache_resource(show_spinner="Loading AI model...")\n'
    'def load_model():\n'
    '    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)\n'
    '    quant_cfg = BitsAndBytesConfig(\n'
    '        load_in_4bit=True,\n'
    '        bnb_4bit_compute_dtype=torch.float16,\n'
    '        bnb_4bit_quant_type="nf4",\n'
    '        bnb_4bit_use_double_quant=True,\n'
    '    )\n'
    '    mdl = AutoModelForCausalLM.from_pretrained(\n'
    '        MODEL_ID,\n'
    '        quantization_config=quant_cfg,\n'
    '        device_map="auto",\n'
    '        low_cpu_mem_usage=True,\n'
    '        trust_remote_code=True,\n'
    '    )\n'
    '    mdl.eval()\n'
    '    return tok, mdl\n'
    '\n'
    'def generate_text(prompt, tok, mdl):\n'
    '    msgs = [{"role": "user", "content": prompt}]\n'
    '    fmt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n'
    '    inputs = tok(fmt, return_tensors="pt").to(mdl.device)\n'
    '    with torch.no_grad():\n'
    '        out = mdl.generate(\n'
    '            **inputs,\n'
    '            max_new_tokens=1200,\n'
    '            do_sample=False,\n'
    '            temperature=None,\n'
    '            top_p=None,\n'
    '            pad_token_id=tok.eos_token_id,\n'
    '            repetition_penalty=1.1,\n'
    '        )\n'
    '    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()\n'
    '    raw = raw.replace("```json", "").replace("```", "").strip()\n'
    '    return raw\n'
    '\n'
    'col1, col2 = st.columns(2)\n'
    'with col1:\n'
    '    subjects   = st.text_area("المواد الدراسية (افصل بينها بفاصلة):", placeholder="مثال: رياضيات، فيزياء، علم حاسوب")\n'
    '    exam_dates = st.text_area("مواعيد الامتحانات التقريبية:", placeholder="مثال: الرياضيات 10 يونيو، الفيزياء 15 يونيو")\n'
    'with col2:\n'
    '    student_level = st.selectbox("ما هو مستواك الحالي في هذه المواد؟", [\n'
    '        "مبتدئ (بحاجة لتأسيس مكثف)",\n'
    '        "متوسط (أحتاج حل وتطبيق)",\n'
    '        "متقدم (مراجعة سريعة وامتحانات)",\n'
    '    ])\n'
    '\n'
    'if st.button("توليد خطة المذاكرة ✨"):\n'
    '    st.session_state.result_df = None\n'
    '    st.session_state.raw_output = None\n'
    '    if subjects and exam_dates:\n'
    '        with st.spinner("جاري تصميم الجدول الذكي..."):\n'
    '            tok, mdl = load_model()\n'
    '            subject_list = [s.strip() for s in subjects.replace("،", ",").split(",") if s.strip()]\n'
    '            total_sessions = len(subject_list) * 4\n'
    '            prompt = (\n'
    '                "Output ONLY a raw JSON object. No markdown, no code fences, no explanation.\\n"\n'
    '                "The JSON must have this exact structure:\\n"\n'
    '                "{\\"plan\\":[{\\"day\\":\\"Day 1 - June 4\\",\\"subject\\":\\"<exact name>\\",\\"task_type\\":\\"<see rules>\\",\\"duration\\":\\"<see rules>\\"}]}\\n"\n'
    '                "Rules:\\n"\n'
    '                "1. subject must be exactly as written in Subjects below\\n"\n'
    '                "2. task_type must vary — rotate through: قراءة المفاهيم, حل تمارين, مراجعة شاملة, حل امتحانات سابقة, مراجعة نهائية\\n"\n'
    '                "3. Mix all subjects across days, do not do one subject at a time\\n"\n'
    '                "4. duration must match task_type: قراءة المفاهيم=1h, حل تمارين=2h, مراجعة شاملة=1.5h, حل امتحانات سابقة=2.5h, مراجعة نهائية=1h\\n"\n'
    '                "5. each subject must appear at least 4 times\\n"\n'
    '                "6. total sessions must be exactly " + str(total_sessions) + "\\n"\n'
    '                "7. JSON must have exactly 4 keys per item: day, subject, task_type, duration — no extra keys\\n"\n'
    '                "8. Output raw JSON only, starting with { and ending with }\\n"\n'
    '                "Subjects: " + subjects + "\\n"\n'
    '                "Exam dates: " + exam_dates + "\\n"\n'
    '                "Level: " + student_level + "\\n"\n'
    '            )\n'
    '            raw = generate_text(prompt, tok, mdl)\n'
    '            result = None\n'
    '            try:\n'
    '                start = raw.index("{")\n'
    '                depth, end = 0, -1\n'
    '                for i, ch in enumerate(raw[start:], start):\n'
    '                    if ch == "{": depth += 1\n'
    '                    elif ch == "}":\n'
    '                        depth -= 1\n'
    '                        if depth == 0:\n'
    '                            end = i\n'
    '                            break\n'
    '                if end != -1:\n'
    '                    result = json.loads(raw[start:end+1])\n'
    '            except (ValueError, json.JSONDecodeError):\n'
    '                result = None\n'
    '            if result and "plan" in result:\n'
    '                df = pd.DataFrame(result["plan"])\n'
    '                rename_map = {"day": "اليوم", "subject": "المادة", "task_type": "نوع المذاكرة", "duration": "المدة المقترحة"}\n'
    '                df = df.rename(columns=rename_map)\n'
    '                keep = ["اليوم", "المادة", "نوع المذاكرة", "المدة المقترحة"]\n'
    '                df = df[[c for c in keep if c in df.columns]]\n'
    '                st.session_state.result_df = df\n'
    '            else:\n'
    '                st.session_state.raw_output = raw\n'
    '    else:\n'
    '        st.warning("الرجاء تعبئة خانة المواد ومواعيد الامتحانات أولاً.")\n'
    '\n'
    'if st.session_state.result_df is not None:\n'
    '    st.success("✅ تم إنشاء خطتك الدراسية بنجاح!")\n'
    '    st.table(st.session_state.result_df)\n'
    'elif st.session_state.raw_output is not None:\n'
    '    st.error("فشل تنسيق JSON — المخرجات الخام:")\n'
    '    st.code(st.session_state.raw_output)\n'
)

with open("/kaggle/working/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py written!")

import py_compile
try:
    py_compile.compile("/kaggle/working/app.py", doraise=True)
    print("Syntax check: PASSED")
except py_compile.PyCompileError as e:
    print("Syntax error:", e)

app.py written!
Syntax check: PASSED


In [10]:
import subprocess, time, socket
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

conf.get_default().auth_token = UserSecretsClient().get_secret("NGROK_TOKEN")
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
time.sleep(2)

subprocess.Popen(["streamlit", "run", "/kaggle/working/app.py",
    "--server.port", "8501", "--server.headless", "true",
    "--server.enableCORS", "false"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def wait(port, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with socket.create_connection(("localhost", port), timeout=1): return True
        except OSError: time.sleep(1)
    return False

if wait(8501): print("Ready!")
else: print("Failed to start")

tunnel = ngrok.connect(8501)
print("Your app:", tunnel.public_url)

Ready!
Your app: https://reviver-reoccupy-scorn.ngrok-free.dev
